In [0]:
%pip install sqlglot -q
%restart_python

In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType, StructType, StructField

# Read the webi_variables table
df_variables = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_variables") \
    .filter(F.col("variable_definition").isNotNull() & (F.col("variable_definition") != ""))

# UDF to extract datafield references from variable_definition
# Patterns: [field_name] or [provider].[field_name]
# Returns list of (provider_qualifier, field_name) tuples
def extract_datafields(definition):
    if not definition:
        return []
    
    results = []
    # Match [X].[Y] pattern (qualified) or standalone [Z] pattern
    # First find all bracket groups
    # Pattern: [provider].[field] or just [field]
    qualified_pattern = r'\[([^\]]+)\]\.\[([^\]]+)\]'
    standalone_pattern = r'\[([^\]]+)\]'
    
    # Find all qualified references first
    qualified_matches = re.findall(qualified_pattern, definition)
    qualified_fields = set()
    for provider, field in qualified_matches:
        results.append((provider, field))
        qualified_fields.add(field)
    
    # Remove qualified patterns from definition to find standalone ones
    cleaned = re.sub(qualified_pattern, '', definition)
    
    # Find standalone [field] references in remaining text
    standalone_matches = re.findall(standalone_pattern, cleaned)
    for field in standalone_matches:
        # Skip string literals and known non-field patterns
        if field not in qualified_fields:
            results.append((None, field))
    
    return results

# Define return schema
schema = ArrayType(StructType([
    StructField("provider_qualifier", StringType(), True),
    StructField("field_name", StringType(), False)
]))

extract_datafields_udf = F.udf(extract_datafields, schema)

# UDF to categorize variable_definition (logic from Variable Dictionary overview)
def categorize_definition(defn):
    if not defn or str(defn).strip() == '':
        return 'Empty'
    d = str(defn).strip().lower()

    # 1. Sum aggregate
    if re.search(r'=\s*sum\s*\(', d):
        return 'Sum Aggregate'
    # 2. Conditional — contains if(...)
    if re.search(r'\bif\s*\(', d):
        return 'Conditional (IF)'
    # 3. FX / Rate conversion
    if re.search(r'(loc\s*:\s*eur|eur\s*:\s*loc|exchange\s*rate|fx\s*rate)', d):
        return 'FX / Rate Conversion'
    # 4. Arithmetic DR/CR
    if re.search(r'\bdr\b.*\bcr\b|\bcr\b.*\bdr\b', d):
        return 'Arithmetic DR/CR'
    # 5. General arithmetic
    if re.search(r'[+\-*/]', d):
        return 'Arithmetic'
    # 6. Simple field reference
    return 'Simple Reference'

categorize_udf = F.udf(categorize_definition, StringType())

# Apply extraction, explode, and categorize
df_extracted = df_variables \
    .withColumn("datafields", extract_datafields_udf(F.col("variable_definition"))) \
    .withColumn("datafield_ref", F.explode("datafields")) \
    .withColumn("Variable_category", categorize_udf(F.col("variable_definition"))) \
    .select(
        F.col("Document_Id"),
        F.col("variable_id"),
        F.col("variable_name"),
        F.col("variable_datatype"),
        F.col("variable_qualification"),
        F.col("variable_definition"),
        F.col("Variable_category"),
        F.col("datafield_ref.provider_qualifier").alias("provider_qualifier"),
        F.col("datafield_ref.field_name").alias("extracted_datafield")
    )
df_extracted = df_extracted.dropDuplicates()
display(df_extracted.filter(F.col("Document_Id") == 412320))

In [0]:
# Read the webi_datadictionary table
df_datadict = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_datadictionary") \
    .select(
        F.col("Document_Id"),
        F.col("Data_Provider_ID"),
        F.col("Data_Provider_Name"),
        F.col("datafield_id"),
        F.col("datafield_name"),
        F.col("datafield_qualification"),
        F.col("DataSource_Type"),
        F.col("DataSource_Name")
    )

# Join on Document_Id and extracted_datafield = datafield_name
# For qualified references [provider].[field], also match on Data_Provider_Name
df_lineage = df_extracted.join(
    df_datadict,
    on=[
        df_extracted["Document_Id"] == df_datadict["Document_Id"],
        df_extracted["extracted_datafield"] == df_datadict["datafield_name"],
        # Match provider qualifier when available, otherwise allow any provider
        (df_extracted["provider_qualifier"].isNull()) |
        (df_extracted["provider_qualifier"] == df_datadict["Data_Provider_Name"])
    ],
    how="left"
).select(
    df_extracted["Document_Id"],
    df_extracted["variable_id"],
    df_extracted["variable_name"],
    df_extracted["variable_datatype"],
    df_extracted["variable_qualification"],
    df_extracted["variable_definition"],
    df_extracted["extracted_datafield"],
    df_extracted["provider_qualifier"],
    df_datadict["Data_Provider_ID"],
    df_datadict["datafield_id"],
    df_datadict["datafield_name"],
    df_datadict["datafield_qualification"],
    df_datadict["DataSource_Type"],
    df_datadict["DataSource_Name"]
)

display(df_lineage.filter(F.col("Document_Id") == 412320))

In [0]:
import sqlglot
from pyspark.sql.types import MapType

# --- 3a: Read CMS, filter out errors, concatenate SQL per Document_Id + Data_Provider_ID ---
df_cms = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms") \
    .filter(
        F.col("SQL_Query").isNotNull() &
        (~F.col("SQL_Query").contains("Error retrieving Query Plan"))
    ) \
    .select("Document_Id", "Data_Provider_ID", "SQL_Index", "SQL_Query")

# Concatenate all SQL_Query parts ordered by SQL_Index per provider
from pyspark.sql.window import Window

df_sql_concat = df_cms \
    .groupBy("Document_Id", "Data_Provider_ID") \
    .agg(
        F.concat_ws(" ", F.collect_list(F.struct(F.col("SQL_Index").cast("int").alias("idx"), F.col("SQL_Query")).getField("SQL_Query"))).alias("full_sql")
    )

# Better approach: sort then concat
w = Window.partitionBy("Document_Id", "Data_Provider_ID").orderBy(F.col("SQL_Index").cast("int"))
df_cms_sorted = df_cms.withColumn("rn", F.row_number().over(w))

df_sql_concat = df_cms_sorted \
    .groupBy("Document_Id", "Data_Provider_ID") \
    .agg(
        F.concat_ws(" ", F.sort_array(F.collect_list(F.struct(F.col("SQL_Index").cast("int"), F.col("SQL_Query"))), asc=True).getField("SQL_Query")).alias("full_sql")
    )

# --- 3b: UDF to parse SQL and map column alias -> source table ---
def parse_sql_field_to_table(full_sql):
    """Parse SQL, return dict mapping column alias (display name) -> 'source_table|||source_column'."""
    if not full_sql or full_sql.strip() == '':
        return {}
    
    # SAP BO preprocessing
    import re
    sql = full_sql
    # Remove @Prompt(...) and @Variable(...) — replace with placeholder
    sql = re.sub(r'@Prompt\([^)]*\)', "'placeholder'", sql)
    sql = re.sub(r'@Variable\([^)]*\)', "'placeholder'", sql)
    # Remove Oracle outer join syntax (+)
    sql = re.sub(r'\(\+\)', '', sql)
    
    try:
        parsed = sqlglot.parse(sql, read="oracle")
    except Exception:
        return {}
    
    field_table_map = {}
    
    for statement in parsed:
        if statement is None:
            continue
        try:
            # Build table alias -> table name mapping from FROM/JOIN
            alias_to_table = {}
            for table in statement.find_all(sqlglot.exp.Table):
                table_name = table.name
                if table.db:
                    table_name = f"{table.db}.{table.name}"
                tbl_alias = table.alias or table.name
                alias_to_table[tbl_alias.lower()] = table_name
            
            # Extract SELECT expressions: find column alias -> source table alias + source column
            select = statement.find(sqlglot.exp.Select)
            if not select:
                continue
            for expr in select.expressions:
                # Get the display alias (the quoted name)
                alias_name = expr.alias
                if not alias_name:
                    continue
                
                # Find the source table alias and column name from the column reference
                col_ref = expr.find(sqlglot.exp.Column)
                if col_ref and col_ref.table:
                    src_table_alias = col_ref.table.lower()
                    src_table = alias_to_table.get(src_table_alias, src_table_alias)
                    src_column = col_ref.name
                    field_table_map[alias_name] = f"{src_table}|||{src_column}"
                elif col_ref:
                    # No table qualifier on column
                    src_column = col_ref.name
                    field_table_map[alias_name] = f"UNQUALIFIED|||{src_column}"
                else:
                    # Expression (literal, function, etc.)
                    field_table_map[alias_name] = "EXPRESSION|||EXPRESSION"
        except Exception:
            continue
    
    return field_table_map

parse_sql_udf = F.udf(parse_sql_field_to_table, MapType(StringType(), StringType()))

# --- 3c: Apply parsing and join with df_lineage ---
df_sql_parsed = df_sql_concat \
    .withColumn("field_table_map", parse_sql_udf(F.col("full_sql")))

# Join lineage with parsed SQL on Document_Id + Data_Provider_ID
df_lineage_sql = df_lineage.join(
    df_sql_parsed,
    on=[
        df_lineage["Document_Id"] == df_sql_parsed["Document_Id"],
        df_lineage["Data_Provider_ID"] == df_sql_parsed["Data_Provider_ID"]
    ],
    how="left"
).select(
    df_lineage["Document_Id"],
    df_lineage["variable_id"],
    df_lineage["variable_name"],
    df_lineage["variable_datatype"],
    df_lineage["variable_qualification"],
    df_lineage["extracted_datafield"],
    df_lineage["provider_qualifier"],
    df_lineage["Data_Provider_ID"],
    df_lineage["datafield_id"],
    df_lineage["datafield_name"],
    df_lineage["datafield_qualification"],
    df_lineage["DataSource_Type"],
    df_lineage["DataSource_Name"],
    df_sql_parsed["full_sql"],
    df_sql_parsed["field_table_map"]
)

# Extract the source table and source column for each datafield_name from the map
df_lineage_sql = df_lineage_sql.withColumn(
    "_lookup",
    F.when(F.col("field_table_map").isNotNull() & F.col("datafield_name").isNotNull(),
           F.col("field_table_map")[F.col("datafield_name")]
    ).otherwise(F.lit(None))
).withColumn(
    "source_table",
    F.split(F.col("_lookup"), "\\|\\|\\|").getItem(0)
).withColumn(
    "source_column",
    F.split(F.col("_lookup"), "\\|\\|\\|").getItem(1)
).drop("_lookup")

display(df_lineage_sql.filter(F.col("Document_Id") == "412320"))
from pyspark.sql import Window

# Get all unique Document_Ids
doc_ids = [row["Document_Id"] for row in df_lineage_sql.select("Document_Id").distinct().collect()]
doc_ids = sorted(doc_ids)
batch_size = 100

for i in range(0, len(doc_ids), batch_size):
    batch_ids = doc_ids[i:i+batch_size]
    df_batch = df_lineage_sql.filter(F.col("Document_Id").isin(batch_ids))
    df_batch.write.mode("overwrite" if i == 0 else "append") \
        .option("overwriteSchema", "true" if i == 0 else "false") \
        .saveAsTable("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_variable_lineage")
    print(f"Finished batch {i} - {i+batch_size}")


In [0]:
from pyspark.sql import functions as F


df = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_variable_lineage")

# Total distinct reports
total_reports = df.select("Document_Id").distinct().count()
total_variables = df.select("Document_Id", "variable_id").distinct().count()
print(f"Total WEBI Reports: {total_reports}")
print(f"Total distinct variables: {total_variables}")
print()

# Variable distribution by category
print("=== Variable Distribution by Category ===")
display(
    df.select("Document_Id", "variable_id", "Variable_category").distinct()
      .groupBy("Variable_category")
      .agg(F.count("*").alias("variable_count"))
      .orderBy(F.desc("variable_count"))
)

In [0]:
%sql
select document_Id, 
  100.0 * count(distinct case when datafield_name is not null and datafield_name != '' then variable_id end) 
    / count(distinct variable_id) as pct_variables_with_data_field
from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_report_variable_lineage
-- where DataSource_Type='fhsql'
group by document_Id
having pct_variables_with_data_field < 100